In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm catboost imbalanced-learn shap openpyxl xlrd jupyter

In [ ]:
import pandas as pd
import numpy as np

import sklearn
import xgboost
import lightgbm
import catboost
import imblearn
import shap

print("Environment Verification")
print("="*50)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-Learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("LightGBM:", lightgbm.__version__)
print("CatBoost:", catboost.__version__)
print("Imbalanced-Learn:", imblearn.__version__)
print("SHAP:", shap.__version__)

In [ ]:
import pandas as pd

file_path = r"C:\Users\gnanesh\Desktop\Fetal_Health\CTG.xls"

raw_df = pd.read_excel(
    file_path,
    sheet_name="Data",
    header=None
)

columns = raw_df.iloc[1].astype(str)

new_cols = []
counts = {}

for col in columns:

    if col in counts:
        counts[col] += 1
        new_cols.append(f"{col}_{counts[col]}")
    else:
        counts[col] = 0
        new_cols.append(col)

df = raw_df.iloc[2:].copy()

df.columns = new_cols

df = df.loc[:, ~pd.isna(df.columns)]

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.reset_index(drop=True, inplace=True)

print("Dataset Loaded Successfully")
print("Shape:", df.shape)

In [ ]:
print("CTG Dataset Assessment")
print("="*50)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nTarget Variable:")
print("NSP")

print("\nTarget Classes:")
print("1 = Normal")
print("2 = Suspect")
print("3 = Pathological")

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nMissing Values:")
print(df.isnull().sum().sum())

In [ ]:
class_counts = df["NSP"].value_counts().sort_index()

print("Class Distribution")
print("="*40)

print("Normal (1):", class_counts[1.0])
print("Suspect (2):", class_counts[2.0])
print("Pathological (3):", class_counts[3.0])

total = len(df)

print("\nPercentages")

for cls, count in class_counts.items():
    print(
        f"Class {int(cls)} : {round((count/total)*100,2)}%"
    )

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))

sns.countplot(
    x=df["NSP"]
)

plt.title(
    "Fetal Health Class Distribution"
)

plt.xlabel("Class")

plt.ylabel("Count")

plt.show()

In [ ]:
minority = class_counts[3.0]
majority = class_counts[1.0]

imbalance_ratio = round(
    majority/minority,
    2
)

print("Class Imbalance Assessment")
print("="*50)

print("Majority Class:", majority)

print("Minority Class:", minority)

print(
    "Imbalance Ratio:",
    imbalance_ratio
)

if imbalance_ratio > 5:
    print(
        "\nSMOTE Recommended"
    )
else:
    print(
        "\nSMOTE Optional"
    )

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0]

missing = missing.sort_values(
    ascending=False
)

print(
    "Columns With Missing Values"
)

missing.head(20)

In [ ]:
plt.figure(
    figsize=(10,6)
)

missing.head(15).plot(
    kind="bar"
)

plt.title(
    "Top Features Containing Missing Values"
)

plt.ylabel(
    "Missing Count"
)

plt.show()

In [ ]:
corr = df.corr(
    numeric_only=True
)

target_corr = (
    corr["NSP"]
    .abs()
    .sort_values(
        ascending=False
    )
)

print(
    target_corr.head(15)
)

In [ ]:
top_features = (
    corr["NSP"]
    .abs()
    .sort_values(
        ascending=False
    )[1:11]
)

plt.figure(figsize=(10,5))

top_features.plot(
    kind="bar"
)

plt.title(
    "Top Features Associated with Fetal Health"
)

plt.ylabel(
    "Absolute Correlation"
)

plt.show()